# 02. フーリエ解析（Colab）

先に `01_train.ipynb` を実行する。

In [ ]:
# セットアップ：Drive のマウント、リポジトリの取得、runs/ を Drive に向ける（再実行しても壊れない）
import os
import subprocess
import sys

from google.colab import drive

REPO_URL = "https://github.com/karenmathtoml-oss/grokking-from-scratch.git"   # ← 自分のものに書き換える
REPO_DIR = "/content/grokking-from-scratch"
RUNS_DIR = "/content/drive/MyDrive/grokking-runs"


def sh(*cmd):
    """コマンドを実行して、出力をノートブックに表示する。失敗したら例外。"""
    result = subprocess.run(cmd, capture_output=True, text=True)
    print((result.stdout + result.stderr).strip())
    result.check_returncode()


drive.mount("/content/drive")

if os.path.isdir(REPO_DIR):
    sh("git", "-C", REPO_DIR, "pull")
else:
    sh("git", "clone", REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

os.makedirs(RUNS_DIR, exist_ok=True)
if not os.path.lexists("runs"):
    os.symlink(RUNS_DIR, "runs")        # runs/ の中身は Drive に残る

sh("pip", "install", "-q", "pytest", "imageio")
sh("git", "log", "-1", "--oneline")     # この実行が、どのコミットのコードかを記録する

## 学習後のスペクトル（ステップ 40000）

In [ ]:
import torch

from grokking.analysis.fourier import embedding_spectrum, neuron_logit_spectrum, top_frequencies
from grokking.config import Config
from grokking.model import Transformer

cfg = Config.load("runs/base/config.json")


def load_model(step: int) -> Transformer:
    """指定したステップのチェックポイントからモデルを復元する。"""
    model = Transformer(cfg.p, cfg.d_model, cfg.n_heads, cfg.d_mlp, seq_len=3, init_seed=cfg.init_seed)
    model.load_state_dict(torch.load(f"runs/base/ckpt_{step:06d}.pt", map_location="cpu"))
    return model


model = load_model(cfg.steps)
spec_E = embedding_spectrum(model.W_E, cfg.p)
spec_L = neuron_logit_spectrum(model.W_U, model.W_out, cfg.p)
print("W_E の上位周波数:", top_frequencies(spec_E))
print("W_L の上位周波数:", top_frequencies(spec_L))

## 学習前（ステップ 0）との比較

学習前のスペクトルは、特定の周波数に偏らないはずです。学習後と並べて違いを確かめます。

In [ ]:
model0 = load_model(0)
spec_E0 = embedding_spectrum(model0.W_E, cfg.p)
print("ステップ 0 の上位周波数:", top_frequencies(spec_E0))
print("最大の強さ（0 / 40000）:", spec_E0.max().item(), "/", spec_E.max().item())

## 学習曲線の確認

In [ ]:
import pandas as pd

df = pd.read_csv("runs/base/metrics.csv")
print(df[df.test_accuracy > 0.5].head(1))
print(df.tail(1))

## 図

図は Drive の `runs/base/figures/` に保存します。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image, display

from grokking.analysis.neurons import mlp_activations
from grokking.analysis.plots import plot_curves, plot_neuron, plot_spectrum, spectrum_gif

FIG_DIR = Path("runs/base/figures")
FIG_DIR.mkdir(exist_ok=True)

### 図1：学習曲線

横軸は対数。訓練精度が先に 1 になり、テスト精度が遅れて跳ねるかを見ます。

In [ ]:
fig = plot_curves("runs/base/metrics.csv", FIG_DIR / "fig1_curve.png")
plt.show()

### 図2：埋め込みスペクトル

まず、ステップ 0 と学習後を並べます。学習後は少数の周波数だけが立つはずです。
右の W_L（ニューロン → ロジット）でも、同じ周波数が立つかを確かめます。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3), sharey=False)
plot_spectrum(spec_E0, title="W_E, step 0", ax=axes[0])
plot_spectrum(spec_E, title=f"W_E, step {cfg.steps}", ax=axes[1])
plot_spectrum(spec_L, title=f"W_L, step {cfg.steps}", ax=axes[2])
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_spectrum_before_after.png", dpi=150)
plt.show()

次に、全チェックポイントのスペクトルを GIF にします。
棒が立ち始める時期と、図1でテスト精度が跳ねる時期のずれを見ます。

In [ ]:
def build_model() -> Transformer:
    """重みを読み込む前の空のモデル。"""
    return Transformer(cfg.p, cfg.d_model, cfg.n_heads, cfg.d_mlp, seq_len=3, init_seed=cfg.init_seed)


spectrum_gif("runs/base", cfg.p, FIG_DIR / "fig2_spectrum.gif", build_model)
display(Image(filename=str(FIG_DIR / "fig2_spectrum.gif")))

### 図3：ニューロンの活性

全組 (a, b) に対する MLP 活性のうち、分散が最大のニューロンを描きます。
縞や格子のような周期的な模様が出れば、(a, b) の周期関数を計算している証拠です。
模様の周期から周波数 k を読み、上の top_frequencies と一致するか確かめます。

In [ ]:
acts = mlp_activations(model)                       # (p, p, d_mlp)
n = acts.var(dim=(0, 1)).argmax().item()
fig = plot_neuron(acts, n, FIG_DIR / "fig3_neuron.png")
plt.show()